# Lab 08 — Algoritmo PID: ações P, I e D e seus efeitos

**Unidade IV — Projeto, sintonia e implementação de PID** · conteúdo 4.1 do PPC

**Objetivos:**
1. Implementar o PID nas formas paralela e ISA;
2. Isolar e demonstrar o efeito de cada ação (P, I, D) na malha fechada;
3. Avaliar o controlador nos **quatro sinais canônicos** (gang of four);
4. Entender a necessidade do filtro derivativo.

**Referências:** Åström & Murray (FBS), cap. 11 · Ogata, cap. 10 · Åström & Hägglund, caps. 1–2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import control as ct
    print("python-control", ct.__version__)
except ImportError:
    %pip install control
    import control as ct

## 1. O PID como função de transferência

Forma ISA: $C(s) = K_p\left(1 + \dfrac{1}{T_i s} + \dfrac{T_d s}{1 + T_d s/N}\right)$ (derivada filtrada).

In [ ]:
def pid_tf(Kp, Ti=np.inf, Td=0.0, N=10):
    """PID na forma ISA com derivada filtrada. Ti=inf desliga o I; Td=0 desliga o D."""
    C = ct.tf([Kp], [1])
    if np.isfinite(Ti):
        C = C + ct.tf([Kp], [Ti, 0])
    if Td > 0:
        C = C + ct.tf([Kp * Td, 0], [Td / N, 1])
    return C

# planta desta aula: 3ª ordem do Lab 05 (Ku = 90, Tu = 1.68 s)
G = ct.tf([1], np.polymul(np.polymul([1, 1], [1, 2]), [1, 4]))
print("Planta:", G)

## 2. Ação proporcional isolada: rapidez × erro × oscilação

In [ ]:
t = np.linspace(0, 10, 1000)
plt.figure(figsize=(9, 5))
for Kp in [5, 20, 60]:
    T_cl = ct.feedback(pid_tf(Kp) * G, 1)
    resp = ct.step_response(T_cl, t)
    e_inf = 1 / (1 + Kp * ct.dcgain(G))
    plt.plot(resp.time, resp.outputs, lw=2, label=f'P: Kp = {Kp} (e_inf = {e_inf:.2f})')
plt.axhline(1, color='gray', ls='--')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Só P: erro de regime persiste; Kp alto oscila')
plt.legend(); plt.grid(True)
plt.show()

## 3. Acrescentando a ação integral: PI

In [ ]:
plt.figure(figsize=(9, 5))
Kp = 20
for Ti in [np.inf, 3.0, 1.0, 0.4]:
    T_cl = ct.feedback(pid_tf(Kp, Ti) * G, 1)
    resp = ct.step_response(T_cl, np.linspace(0, 12, 1200))
    rotulo = f'PI: Ti = {Ti}' if np.isfinite(Ti) else 'P puro'
    plt.plot(resp.time, resp.outputs, lw=2, label=rotulo)
plt.axhline(1, color='gray', ls='--')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Ação I zera o erro; Ti pequeno demais desestabiliza')
plt.legend(); plt.grid(True)
plt.show()

**Trade-off da ação integral:** $T_i$ menor ⟹ erro zerado mais rápido, porém o integrador
adiciona atraso de fase (−90°) e **corrói a margem de fase** — a oscilação cresce até instabilizar.

In [ ]:
# margens de fase em função de Ti (auditoria da sintonia)
for Ti in [3.0, 1.0, 0.4]:
    gm, pm, _, _ = ct.margin(pid_tf(20, Ti) * G)
    print(f"Ti = {Ti:4.1f} -> PM = {pm:5.1f} graus | GM = {20*np.log10(gm):5.1f} dB")

## 4. Acrescentando a ação derivativa: PID completo

In [ ]:
plt.figure(figsize=(9, 5))
Kp, Ti = 20, 1.0
for Td in [0.0, 0.15, 0.4]:
    T_cl = ct.feedback(pid_tf(Kp, Ti, Td) * G, 1)
    resp = ct.step_response(T_cl, np.linspace(0, 12, 1200))
    plt.plot(resp.time, resp.outputs, lw=2,
             label=f'Td = {Td}' + (' (PI)' if Td == 0 else ''))
plt.axhline(1, color='gray', ls='--')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Ação D amortece: recupera a margem que o I consumiu')
plt.legend(); plt.grid(True)
plt.show()

for Td in [0.0, 0.15, 0.4]:
    gm, pm, _, _ = ct.margin(pid_tf(20, 1.0, Td) * G)
    print(f"Td = {Td:4.2f} -> PM = {pm:5.1f} graus")

A derivada **adianta fase** (como um zero), amortecendo a resposta e permitindo ganhos maiores.
O preço aparece no ruído:

## 5. Por que a derivada precisa de filtro

In [ ]:
# malha com ruído de medição: comparar esforço de controle com N = 5, 10, 100
rng = np.random.default_rng(3)
t_n = np.linspace(0, 8, 4000)
r_n = np.ones_like(t_n)

plt.figure(figsize=(9, 5))
for N in [5, 20, 200]:
    C = pid_tf(20, 1.0, 0.3, N=N)
    # função r,n -> u : u = C S (r - n); avaliamos o ramo do ruído n -> u
    S = ct.feedback(1, C * G)        # sensibilidade
    Gun = -C * S                     # ruído de medição -> controle
    ruido = rng.normal(0, 0.01, size=t_n.shape)   # ruído de sensor (sigma = 1%)
    resp_u = ct.forced_response(Gun, t_n, ruido)
    plt.plot(resp_u.time, resp_u.outputs, lw=0.8, label=f'N = {N}')
plt.xlabel('Tempo [s]'); plt.ylabel('u(t) devido apenas ao ruído')
plt.title('Sem filtro (N grande), a derivada amplifica o ruído no atuador')
plt.legend(); plt.grid(True)
plt.show()

Com N = 200 (derivada quase pura), o ruído de 1 % do sensor vira atividade violenta no atuador —
desgaste mecânico e aquecimento. **N entre 8 e 20 é o padrão industrial.**

## 6. A "gang of four": avaliação completa da malha

Uma malha bem projetada deve ser avaliada em 4 respostas (Åström & Murray, cap. 12):
seguimento de referência ($T$), rejeição de perturbação na entrada da planta ($GS$),
esforço devido ao ruído ($CS$) e sensibilidade ($S$).

In [ ]:
C = pid_tf(20, 1.0, 0.3, N=10)
L = C * G
S = ct.feedback(1, L)      # sensibilidade
T_c = ct.feedback(L, 1)    # complementar (r -> y)
GS = G * S                 # perturbação de entrada -> saída
CS = C * S                 # ruído -> controle

t4 = np.linspace(0, 10, 1000)
fig, axs = plt.subplots(2, 2, figsize=(11, 7))
for ax, sys, titulo in [(axs[0, 0], T_c, 'T: degrau de referência → y'),
                        (axs[0, 1], GS, 'GS: degrau de perturbação → y'),
                        (axs[1, 0], CS, 'CS: degrau de ruído → u'),
                        (axs[1, 1], S, 'S: degrau de referência → erro')]:
    resp = ct.step_response(sys, t4)
    ax.plot(resp.time, resp.outputs, lw=2)
    ax.set_title(titulo); ax.grid(True)
fig.suptitle('Gang of four: as quatro faces da mesma malha')
plt.tight_layout()
plt.show()

# pico da sensibilidade: medida escalar de robustez (bom: Ms < 2)
mag = ct.frequency_response(S, np.logspace(-2, 2, 500)).magnitude.squeeze()
print(f"Pico de sensibilidade Ms = {mag.max():.2f}  (critério de robustez: Ms <= 2)")

`python-control` traz a versão em frequência pronta em uma linha — use-a como verificação
padrão em todo projeto (é o gráfico de auditoria do CDS 110):

In [ ]:
ct.gangof4(G, C)
plt.show()

## 7. Por que o "I" é inegociável: robustez a erro de modelo

Há uma razão mais profunda para a ação integral do que "zerar erro de regime para a planta
nominal": **ela zera o erro mesmo quando o modelo está errado** (desde que a malha continue
estável). Um controlador estático calibrado no modelo nominal perde o ponto de operação
quando a planta muda; o integrador o recupera automaticamente. Demonstração — controlador P
e PI projetados para $G$ nominal, aplicados a uma planta com **ganho 30 % maior e polo
deslocado**:

In [ ]:
G_real = ct.tf([1.3], np.polymul(np.polymul([1, 0.8], [1, 2]), [1, 4]))  # planta "de verdade"

C_P = pid_tf(20)              # so proporcional
C_PI = pid_tf(20, 1.0)        # proporcional + integral

t7 = np.linspace(0, 10, 1000)
plt.figure(figsize=(9, 5))
for C_k, nome, cor in [(C_P, 'P', 'C3'), (C_PI, 'PI', 'C2')]:
    y_nom = ct.step_response(ct.feedback(C_k * G, 1), t7).outputs
    y_real = ct.step_response(ct.feedback(C_k * G_real, 1), t7).outputs
    plt.plot(t7, y_nom, cor, ls='--', lw=1.5, label=f'{nome} na planta nominal')
    plt.plot(t7, y_real, cor, lw=2.5, label=f'{nome} na planta perturbada')
plt.axhline(1, color='gray', ls=':')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Erro de modelo: o P muda o valor final; o PI sempre retorna a r = 1')
plt.legend(); plt.grid(True)
plt.show()

O controlador P entrega valores finais diferentes nas duas plantas (o erro de regime depende
do ganho de malha, que mudou). O PI converge para $y = 1$ **nos dois casos** — o integrador
só descansa quando $e = 0$, seja qual for a planta. É por isso que praticamente toda malha
industrial carrega ação integral, e por que erros de identificação no projeto final não
comprometem o valor de regime (só o transitório).

---
> **🖼️ Figuras de apoio nos livros:**
> - Ogata, **Figura 8.1** — controle PID de uma planta (diagrama de blocos). Cap. 8, §8.2, **p. 522** (p. 533 do PDF).
> - Transparências CDS 110 **L9-1**, **slides 6–8** — efeito de P, PI e PID no Bode e na resposta ao degrau.
> - Transparências CDS 110 **L9-1**, **slide 9** — 'Implementing Derivative Action': filtro do D e derivada fora do ramo da referência.

## Exercícios (relatório do Lab 08)

**E1.** Para a planta do laboratório, ajuste manualmente ($K_p$, $T_i$, $T_d$) buscando
$M_p \le 10\,\%$ e $t_s \le 3$ s. Documente o processo (que botão mexeu, o que observou) e
reporte as margens finais.

**E2.** Mostre que o controlador PD (sem I) não zera o erro de regime nesta planta, mas
calcule para qual classe de plantas o P puro já zera o erro ao degrau (dica: plantas com
integrador).

**E3.** Trace o Bode do PID com e sem filtro derivativo (N = 10 vs derivada pura) e explique
a diferença em altas frequências.

**E4.** Reproduza a gang of four para sua sintonia do E1 e verifique $M_s \le 2$. Se violou,
re-sintonize.

In [ ]:
# E1 — sua solução aqui

In [ ]:
# E2 — sua solução aqui

In [ ]:
# E3 — sua solução aqui

In [ ]:
# E4 — sua solução aqui